In [27]:
from src.core.parser import parse_data
from src.core.preprocessor import preprocess
from pathlib import Path
from collections import Counter
import numpy as np
import json





# Phase1: Classes in rooms - Loading Data

In [4]:

genome = np.load('results/phase1_6000_best.npy')
n_classes = genome.shape[0]
subset = n_classes if n_classes < 896 else None
tt = parse_data('data/data.xml', subset=subset)
pp = preprocess(tt)

exempt = getattr(pp, 'room_share_exempt_pairs', getattr(pp, 'meet_with_exempt_pairs', set()))
print(f'loaded {len(tt.classes)} classes, genome size {genome.shape}')



loaded 896 classes, genome size (896, 2)


# Finding conflicts

In [7]:
classes_by_room = {}
for ci, cls in enumerate(tt.classes):
    if not cls.candidate_rooms:
        continue
    room_idx = int(genome[ci, 0])
    room_id = cls.candidate_rooms[room_idx].room_id
    classes_by_room.setdefault(room_id, []).append(ci)

conflicts = []
for room_id, class_idxs, in classes_by_room.items():
    if len(class_idxs) < 2:
        continue
    for i in range(len(class_idxs)):
        for j in range(i + 1, len(class_idxs)):
            ci_a, ci_b = class_idxs[i], class_idxs[j]

        key = (min(ci_a, ci_b), max(ci_a, ci_b))
        if key in exempt:
            continue

        mask_a = pp.time_masks[ci_a][int(genome[ci_a, 1])]
        mask_b = pp.time_masks[ci_b][int(genome[ci_b, 1])]
        if np.any(mask_a & mask_b):
            cls_a = tt.classes[ci_a]
            cls_b = tt.classes[ci_b]
            tp_a = cls_a.candidate_times[int(genome[ci_a, 1])]
            tp_b = cls_b.candidate_times[int(genome[ci_b, 1])]
            room = tt.rooms_by_id[room_id]

            conflicts.append({
                    'room_id':   room_id,
                    'room_cap':  room.capacity,
                    'class_a':   cls_a.id,
                    'class_b':   cls_b.id,
                    'dept_a':    cls_a.department,
                    'dept_b':    cls_b.department,
                    'days_a':    tp_a.days,
                    'start_a':   tp_a.start,
                    'length_a':  tp_a.length,
                    'days_b':    tp_b.days,
                    'start_b':   tp_b.start,
                    'length_b':  tp_b.length,
                })
print(f'found {len(conflicts)} conflicts')

found 5 conflicts


## Listing and plotting conflicts

In [26]:
for c in conflicts:
    cls_a = tt.classes_by_id[c['class_a']]
    cls_b = tt.classes_by_id[c['class_b']]

    both_pinned = (len(cls_a.candidate_rooms) == 1 and
                   len(cls_a.candidate_times) == 1 and
                   len(cls_b.candidate_rooms) == 1 and
                   len(cls_b.candidate_times) == 1)

    c['fixable'] = 'PERMANENT' if both_pinned else 'FIXABLE'
    c['opts_a'] = f"{len(cls_a.candidate_rooms)}r × {len(cls_a.candidate_times)}t"
    c['opts_b'] = f"{len(cls_b.candidate_rooms)}r × {len(cls_b.candidate_times)}t"

DAYS_SHORT = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

def slot_to_time(slot):
    minutes = slot * 5
    h, m = divmod(minutes, 60)
    ampm = 'am' if h < 12 else 'pm'
    h12 = h if 1 <= h <= 12 else (h - 12 if h > 12 else 12)
    return f'{h12}:{m:02d}{ampm}'

def format_days(days_mask):
    return '/'.join(d for i, d in enumerate(DAYS_SHORT) if days_mask & (1 << i))

for c in conflicts:
    cls_a = tt.classes_by_id[c['class_a']]
    cls_b = tt.classes_by_id[c['class_b']]

    both_pinned = (len(cls_a.candidate_rooms) == 1 and
                   len(cls_a.candidate_times) == 1 and
                   len(cls_b.candidate_rooms) == 1 and
                   len(cls_b.candidate_times) == 1)

    status = 'PERMANENT' if both_pinned else 'FIXABLE'

    time_a = f"{format_days(c['days_a'])} {slot_to_time(c['start_a'])}–{slot_to_time(c['start_a'] + c['length_a'])}"
    time_b = f"{format_days(c['days_b'])} {slot_to_time(c['start_b'])}–{slot_to_time(c['start_b'] + c['length_b'])}"

    print(f"[{status}] Room {c['room_id']} (cap {c['room_cap']})")
    print(f"  Class {c['class_a']}  {time_a}  opts: {len(cls_a.candidate_rooms)}r x {len(cls_a.candidate_times)}t")
    print(f"  Class {c['class_b']}  {time_b}  opts: {len(cls_b.candidate_rooms)}r x {len(cls_b.candidate_times)}t")
    print()

    n_permanent = sum(1 for c in conflicts
                  if tt.classes_by_id[c['class_a']].candidate_rooms.__len__() == 1 and
                     tt.classes_by_id[c['class_a']].candidate_times.__len__() == 1 and
                     tt.classes_by_id[c['class_b']].candidate_rooms.__len__() == 1 and
                     tt.classes_by_id[c['class_b']].candidate_times.__len__() == 1)

print(f'Total:     {len(conflicts)}')
print(f'Permanent: {n_permanent}')
print(f'Fixable:   {len(conflicts) - n_permanent}')

[FIXABLE] Room 14 (cap 268)
  Class 613  Mon/Wed/Fri 9:30am–10:30am  opts: 11r x 10t
  Class 808  Mon/Wed/Fri 9:30am–10:30am  opts: 6r x 10t

[PERMANENT] Room 37 (cap 70)
  Class 694  Fri 12:30pm–1:30pm  opts: 1r x 1t
  Class 749  Fri 8:00am–5:10pm  opts: 1r x 1t

[PERMANENT] Room 37 (cap 70)
  Class 739  Fri 11:30am–12:30pm  opts: 1r x 1t
  Class 749  Fri 8:00am–5:10pm  opts: 1r x 1t

[PERMANENT] Room 37 (cap 70)
  Class 746  Fri 3:30pm–5:30pm  opts: 1r x 1t
  Class 749  Fri 8:00am–5:10pm  opts: 1r x 1t

[PERMANENT] Room 37 (cap 70)
  Class 746  Fri 3:30pm–5:30pm  opts: 1r x 1t
  Class 749  Fri 8:00am–5:10pm  opts: 1r x 1t

Total:     5
Permanent: 4
Fixable:   1


# Phase 2: Students placed into built schedule

## Summary

In [24]:
assignment_path = sorted(Path('results').glob('phase2_full_assignment.json'))[-1]
phase2_data = json.load(open(assignment_path))

print(f"Students processed:  {phase2_data['n_students_processed']:,}")
print(f"Enrollments placed:  {phase2_data['n_enrollments_placed']:,}")
print(f"Enrollments skipped: {phase2_data['n_enrollments_skipped']:,}")
print(f"Coverage:            {phase2_data['coverage_pct']:.1f}%")

Students processed:  30,846
Enrollments placed:  84,495
Enrollments skipped: 7,794
Coverage:            91.6%


## Total Skipped and why skipped

In [22]:


assignment = phase2_data['assignment']
skipped = []

for student in tt.students:
    sid_str = str(student.id)
    student_assignments = assignment.get(sid_str, {})

    for enrollment in student.enrollments:
        off_id_str = str(enrollment.offering_id)

        if off_id_str not in student_assignments:

            sections = []
            for cls in tt.classes:
                if cls.offering == enrollment.offering_id:
                    sections.append(cls)

            if not sections:
                reason = 'no sections in data'
            else:

                section_enrollments = phase2_data['section_enrollments']
                any_capacity = any(
                    section_enrollments.get(str(cls.id), 0) < cls.class_limit
                    for cls in sections
                )
                reason = 'all sections full' if not any_capacity else 'time conflict with other classes'

            skipped.append({
                'student_id':  student.id,
                'offering_id': enrollment.offering_id,
                'n_sections':  len(sections),
                'reason':      reason,
            })

print(f'\nTotal skipped enrollments found: {len(skipped)}')
reason_counts = Counter(s['reason'] for s in skipped)
print('Skipped by reason:')
for reason, count in reason_counts.most_common():
    print(f'  {reason}: {count}')

print()

# Print the first n_shown in detail
n_shown = 30
for s in skipped[:n_shown]:
    print(f"  Student {s['student_id']:6d}  Offering {s['offering_id']:4d}  "
          f"({s['n_sections']} sections)  [{s['reason']}]")

if len(skipped) > 30:
    print(f'  ... and {len(skipped) - 30} more')


Total skipped enrollments found: 7794
Skipped by reason:
  time conflict with other classes: 4848
  all sections full: 2946

  Student      4  Offering  202  (6 sections)  [time conflict with other classes]
  Student     11  Offering  317  (1 sections)  [time conflict with other classes]
  Student     12  Offering  497  (1 sections)  [time conflict with other classes]
  Student     16  Offering  365  (1 sections)  [all sections full]
  Student     21  Offering  202  (6 sections)  [time conflict with other classes]
  Student     23  Offering  202  (6 sections)  [time conflict with other classes]
  Student     27  Offering  303  (3 sections)  [all sections full]
  Student     32  Offering  611  (1 sections)  [all sections full]
  Student     35  Offering    8  (1 sections)  [time conflict with other classes]
  Student     37  Offering  202  (6 sections)  [time conflict with other classes]
  Student     48  Offering  407  (2 sections)  [all sections full]
  Student     49  Offering  223 

## Student Time Conflicts

In [19]:
class_idx_by_id = {c.id: i for i, c in enumerate(tt.classes)}

student_conflicts = []

for sid_str, offerings in assignment.items():
    all_class_ids = [
        cid
        for class_list in offerings.values()
        for cid in class_list
    ]

    # Get the time mask for each
    masks = []
    for cid in all_class_ids:
        ci = class_idx_by_id.get(cid)
        if ci is None:
            continue
        time_idx = int(genome[ci, 1])
        if time_idx < pp.time_masks[ci].shape[0]:
            masks.append((cid, pp.time_masks[ci][time_idx]))

    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            cid_a, mask_a = masks[i]
            cid_b, mask_b = masks[j]
            if np.any(mask_a & mask_b):
                student_conflicts.append({
                    'student_id': int(sid_str),
                    'class_a': cid_a,
                    'class_b': cid_b,
                })

print(f'\nStudent time conflicts: {len(student_conflicts)}')
if student_conflicts:
    for sc in student_conflicts[:20]:
        print(f"  Student {sc['student_id']}  Class {sc['class_a']} vs Class {sc['class_b']}")
else:
    print('  None — sectioner correctly avoided all time conflicts')


Student time conflicts: 0
  None — sectioner correctly avoided all time conflicts


## Section analysis

In [21]:
section_enrollments = phase2_data['section_enrollments']

print('\nSection fill analysis:')
overfull  = []
empty     = []
underfull = []

for cls in tt.classes:
    enrolled = int(section_enrollments.get(str(cls.id), 0))
    limit = cls.class_limit

    if enrolled > limit:
        overfull.append((cls.id, enrolled, limit))
    elif enrolled == 0:
        empty.append((cls.id, limit))
    elif enrolled < limit * 0.5:
        underfull.append((cls.id, enrolled, limit))

print(f'  Overfull sections: {len(overfull)}')
print(f'  Empty sections:                 {len(empty)}')
print(f'  Under 50% full:                 {len(underfull)}')

if overfull:
    print('\n  OVERFULL (should not happen):')
    for cid, enrolled, limit in overfull:
        print(f'    Class {cid}: {enrolled} students, limit {limit}')


print('\nTop 10 most enrolled sections:')
top_sections = sorted(
    ((int(cid), int(count)) for cid, count in section_enrollments.items()),
    key=lambda x: -x[1]
)[:10]

for cid, count in top_sections:
    cls = tt.classes_by_id[cid]
    pct = count / cls.class_limit * 100
    print(f'  Class {cid:4d}  {count:4d}/{cls.class_limit} students  ({pct:.0f}% full)')


Section fill analysis:
  Overfull sections: 0
  Empty sections:                 70
  Under 50% full:                 48

Top 10 most enrolled sections:
  Class  754   479/600 students  (80% full)
  Class  877   470/470 students  (100% full)
  Class  381   468/468 students  (100% full)
  Class  806   468/474 students  (99% full)
  Class  840   468/470 students  (100% full)
  Class  842   466/468 students  (100% full)
  Class  804   462/468 students  (99% full)
  Class  192   460/468 students  (98% full)
  Class  736   448/474 students  (95% full)
  Class  802   444/450 students  (99% full)
